# Closim on a Colab T4

Clone + build + fast CPU tests + Anny (Apache-2.0) avatar bake.
Run top-to-bottom on a GPU runtime (Runtime → Change runtime type → T4 GPU).
Each cell is idempotent — re-running the notebook re-clones fresh.

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
# Node 20 (Colab images ship without node/npm)
import shutil
if shutil.which('node') is None:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt-get install -y nodejs
!node --version && npm --version

In [ ]:
import os
REPO = 'https://github.com/mohdabraralikhan/closim.git'
BRANCH = 'master'
WORK = '/content/closim'
OUT = '/content/closim-out'
os.makedirs(OUT, exist_ok=True)
if os.path.isdir(os.path.join(WORK, '.git')):
    print('repo exists, pulling')
    get_ipython().system(f'git -C {WORK} fetch origin && git -C {WORK} checkout {BRANCH} && git -C {WORK} pull --ff-only')
else:
    get_ipython().system(f'git clone --branch {BRANCH} --depth 1 {REPO} {WORK}')
get_ipython().system(f'git -C {WORK} log --oneline -3')

In [ ]:
%cd /content/closim
!npm ci --no-audit --no-fund || npm install --no-audit --no-fund
!npm run build  # tsc must be clean

In [ ]:
# Fast CPU subset only (no GPU / adversarial / heavy-sim files).
!npx vitest run tests/smoke.test.ts tests/garment/avatar-mesh.test.ts tests/garment/sewing.test.ts tests/production/workflow.test.ts tests/cad/g13-adversarial.test.ts tests/construction/g16-workflow.test.ts

In [ ]:
!pip install --quiet anny
import anny
probe = anny.Anny()
print('phenotypes:', list(probe.phenotype_labels))
print('bones:', probe.bone_count)

In [ ]:
import json
import torch, anny, trimesh

model = anny.Anny().to(dtype=torch.float32)
phen = {key: 0.5 for key in model.phenotype_labels}

def set_like(*needles, value):
    for key in phen:
        if all(n in key.lower() for n in needles):
            phen[key] = value

set_like('gender', value=0.0)  # 0 = female end (verify against labels above)
set_like('female', value=1.0)
set_like('male', value=0.0)
set_like('age', value=0.45)
set_like('baby', value=0.0)
set_like('child', value=0.0)
set_like('height', value=0.55)
set_like('weight', value=0.45)
set_like('muscle', value=0.45)

pose = torch.eye(4)[None, None].repeat(1, model.bone_count, 1, 1)
out = model(pose_parameters=pose, phenotype_kwargs=phen)
verts = out['vertices'].squeeze(dim=0).detach().cpu().numpy()
faces = model.faces
print(f'raw verts={verts.shape} faces={faces.shape}')
print(f'raw bounds min={verts.min(axis=0).tolist()} max={verts.max(axis=0).tolist()}')

height = float(verts[:, 1].max() - verts[:, 1].min())
scale = 0.01 if height > 50 else (0.1 if height > 5 else 1.0)
verts = verts * scale
print(f'height_raw={height:.3f} scale={scale} height_m={height * scale:.3f}')
assert 1.4 <= height * scale <= 2.0, 'avatar height outside sane range'

verts[:, 1] -= verts[:, 1].min()  # feet on y=0
verts[:, 0] -= (verts[:, 0].min() + verts[:, 0].max()) / 2
verts[:, 2] -= (verts[:, 2].min() + verts[:, 2].max()) / 2

OBJ_OUT = '/content/closim-out/closim-avatar-female-168.obj'
mesh = trimesh.Trimesh(vertices=verts, faces=faces)
mesh.export(OBJ_OUT)
stats = {
    'verts': int(len(verts)),
    'tris': int(len(faces)),
    'height_m': float(verts[:, 1].max()),
    'watertight': bool(mesh.is_watertight),
    'phenotype': phen,
}
with open('/content/closim-out/closim-avatar-female-168.stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

## Download

Files → `/content/closim-out/` → download `closim-avatar-female-168.obj` (+ stats JSON) into the repo's `assets/` folder, then validate locally:

```bash
npx tsx tools/avatar-check.mts ./assets/closim-avatar-female-168.obj
```